# Text Generation with LLaDA-8B-Instruct

## Trustworthy AI — Watermarking Diffusion Language Models

This notebook reproduces the text generation procedure using
LLaDA-8B-Instruct and the watermarking implementation proposed
in *Watermarking Diffusion Language Models* (Gloaguen et al., 2026).

### Experimental setup

- Model: GSAI-ML/LLaDA-8B-Instruct
- Watermark: DiffusionKGW_Optimal_Gaussian (Ours)
- Datasets: Longform QA, Finance QA, AlpacaFarm (WaterBench)
- Samples: 200 prompts per dataset
- Conditions: Watermarked and Unwatermarked
- Total: 1,200 generated responses

### Generation strategy

Each dataset is divided into 10 batches of 20 prompts.

For readability, this notebook demonstrates the procedure using
batches 01–05 of Longform QA. The same procedure was repeated
for all remaining batches and datasets.

Results are stored as JSONL files for subsequent watermark
detection, robustness evaluation, and attack experiments.

### Reference

https://github.com/eth-sri/diffusion-lm-watermark

## 1. Environment Setup

The experiments are conducted on Google Colab, using a CPU runtime
for environment preparation and an NVIDIA A100 GPU for text generation.

We use Python 3.12 and install the dependencies provided by
the official implementation.

The repository is cloned into the Colab runtime, and a dedicated
Python virtual environment is created using `uv`.

In [1]:
from pathlib import Path
import subprocess

REPO = Path("/content/diffusion-lm-watermark")
VENV = Path("/content/.venv")
PYTHON = VENV / "bin/python"

# Create a Python 3.12 virtual environment
if not PYTHON.exists():
    subprocess.run(
        ["uv", "venv", str(VENV), "--python", "3.12", "--seed"],
        check=True
    )

# Clone the official repository
if not (REPO / ".git").exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/eth-sri/diffusion-lm-watermark.git",
            str(REPO)
        ],
        check=True
    )

# Install dependencies
subprocess.run(
    [
        "uv", "pip", "install",
        "-r", str(REPO / "requirements.txt"),
        "--python", str(PYTHON),
        "--torch-backend=auto"
    ],
    check=True
)

# Install the watermarking package
subprocess.run(
    [
        "uv", "pip", "install",
        "-e", str(REPO),
        "--python", str(PYTHON)
    ],
    check=True
)

print("Environment setup completed.")

Environment setup completed.


## 2. Environment Verification

Before generating text, we verify that:

1. CUDA is available.
2. The GPU is correctly detected.
3. The official WaterBench datasets are accessible.
4. The repository revision is recorded for reproducibility.

In [2]:
import subprocess
from pathlib import Path

REPO = Path("/content/diffusion-lm-watermark")
PYTHON = Path("/content/.venv/bin/python")

# GPU and PyTorch verification
result = subprocess.run(
    [
        str(PYTHON), "-c",
        "import torch; "
        "print('CUDA available:', torch.cuda.is_available()); "
        "print('GPU:', torch.cuda.get_device_name(0) "
        "if torch.cuda.is_available() else 'None')"
    ],
    check=True,
    capture_output=True,
    text=True
)

print(result.stdout)


# Record the repository revision
commit = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"],
    text=True
).strip()

print("Repository commit:", commit)

# Dataset verification
datasets = [
    "2-1_longform_qa",
    "2-2_finance_qa",
    "5-1_alpacafarm"
]

for dataset in datasets:
    path = REPO / "data/WaterBench" / f"{dataset}.jsonl"
    assert path.is_file(), f"Missing dataset: {path}"
    print(f"Dataset available: {dataset}")

CUDA available: False
GPU: None

Repository commit: 380a4bef75879c28ebcfaf954f1974deb0d54980
Dataset available: 2-1_longform_qa
Dataset available: 2-2_finance_qa
Dataset available: 5-1_alpacafarm


## 3. Dataset Preparation

We select the first 200 prompts from each of three WaterBench datasets:
Longform QA, Finance QA, and AlpacaFarm.

Each dataset is divided into 10 batches of 20 prompts,
resulting in 30 batches in total.

This batching strategy enables incremental generation and
result preservation across Colab sessions.

The original prompt records are preserved without modification.

In [3]:
import json
from pathlib import Path

REPO = Path("/content/diffusion-lm-watermark")

DATASETS = [
    "2-1_longform_qa",
    "2-2_finance_qa",
    "5-1_alpacafarm"
]

NUM_PROMPTS = 200
BATCH_SIZE = 20
BATCH_DIR = Path("/content/watermark_batches")

BATCH_DIR.mkdir(parents=True, exist_ok=True)

for dataset in DATASETS:
    source = REPO / "data/WaterBench" / f"{dataset}.jsonl"

    with source.open(encoding="utf-8") as f:
        prompts = [
            json.loads(line)
            for line in f
            if line.strip()
        ][:NUM_PROMPTS]

    assert len(prompts) == NUM_PROMPTS
    assert all("input" in row and "context" in row for row in prompts)

    for start in range(0, NUM_PROMPTS, BATCH_SIZE):
        batch_id = start // BATCH_SIZE + 1
        selected = prompts[start:start + BATCH_SIZE]

        batch_path = BATCH_DIR / f"{dataset}_batch_{batch_id:02d}.jsonl"

        with batch_path.open("w", encoding="utf-8") as f:
            for row in selected:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")

        print(f"{batch_path.name}: {len(selected)} prompts")

print("Dataset preparation completed.")

2-1_longform_qa_batch_01.jsonl: 20 prompts
2-1_longform_qa_batch_02.jsonl: 20 prompts
2-1_longform_qa_batch_03.jsonl: 20 prompts
2-1_longform_qa_batch_04.jsonl: 20 prompts
2-1_longform_qa_batch_05.jsonl: 20 prompts
2-1_longform_qa_batch_06.jsonl: 20 prompts
2-1_longform_qa_batch_07.jsonl: 20 prompts
2-1_longform_qa_batch_08.jsonl: 20 prompts
2-1_longform_qa_batch_09.jsonl: 20 prompts
2-1_longform_qa_batch_10.jsonl: 20 prompts
2-2_finance_qa_batch_01.jsonl: 20 prompts
2-2_finance_qa_batch_02.jsonl: 20 prompts
2-2_finance_qa_batch_03.jsonl: 20 prompts
2-2_finance_qa_batch_04.jsonl: 20 prompts
2-2_finance_qa_batch_05.jsonl: 20 prompts
2-2_finance_qa_batch_06.jsonl: 20 prompts
2-2_finance_qa_batch_07.jsonl: 20 prompts
2-2_finance_qa_batch_08.jsonl: 20 prompts
2-2_finance_qa_batch_09.jsonl: 20 prompts
2-2_finance_qa_batch_10.jsonl: 20 prompts
5-1_alpacafarm_batch_01.jsonl: 20 prompts
5-1_alpacafarm_batch_02.jsonl: 20 prompts
5-1_alpacafarm_batch_03.jsonl: 20 prompts
5-1_alpacafarm_batch_04.

## 4. Model and Watermark Configuration

We use LLaDA-8B-Instruct with the generation parameters
specified in the official implementation.

### Model parameters

| Parameter | Value |
|---|---|
| Model | GSAI-ML/LLaDA-8B-Instruct |
| Steps | 300 |
| Generation length | 300 |
| Temperature | 0.5 |
| Remasking | random |
| Block length | 25 |
| CFG scale | 0.0 |
| Precision | bfloat16 |

### Watermark parameters

| Parameter | Value |
|---|---|
| Watermark type | Ours |
| Delta | 4.0 |
| Gamma | 0.25 |
| Top-k | 50 |
| Seeding scheme | sumhash |
| Greenlist type | bernoulli |
| Iterations | 1 |

We evaluate two experimental conditions:

1. Watermarked generation using the proposed watermark.
2. Unwatermarked generation using the same model and
   generation parameters.

We use delta = 4.0 instead of delta = 2.0 in the official
reference configuration. This choice must be considered when
comparing our results with those reported by the authors.

In [4]:
import yaml
from pathlib import Path



BATCH_DIR = Path("/content/watermark_batches")
CONFIG_DIR = Path("/content/production_configs")

CONFIG_DIR.mkdir(parents=True, exist_ok=True)

MODEL_CONFIG = {
    "model_name": "GSAI-ML/LLaDA-8B-Instruct",
    "tokenizer_name": "GSAI-ML/LLaDA-8B-Instruct",
    "torch_dtype": "bfloat16",
    "device_map": "cuda",
    "trust_remote_code": True,
    "steps": 300,
    "gen_length": 300,
    "temperature": 0.5,
    "remasking": "random",
    "model_specific_arguments": {
        "cfg_scale": 0.0,
        "mask_id": 126336,
        "block_length": 25
    }
}

WM_CONFIG = {
    "watermark_type": "DiffusionKGW_Optimal_Gaussian",
    "delta": 4.0,
    "convolution_kernel": [-1],
    "topk": 50,
    "seeding_scheme": "sumhash",
    "greenlist_type": "bernoulli",
    "greenlist_params": {
        "gamma": 0.25
    },
    "enforce_kl": False,
    "n_iter": 1
}

NO_WM_CONFIG = {
    "watermark_type": "None"
}

PROMPT_TEMPLATES = {
    "2-1_longform_qa": (
        "You are a helpful assistant, please answer the "
        "following question within 300 words:\n"
        "{context}\n{input}"
    ),
    "2-2_finance_qa": (
        "You are a helpful assistant, please answer the "
        "following question with financial knowledge "
        "within 300 words:\n{context}\n{input}"
    ),
    "5-1_alpacafarm": (
        "You are a helpful assistant, please answer "
        "the following instruction: \n{context}\n{input}"
    )
}

for dataset in PROMPT_TEMPLATES:
    for batch_id in range(1, 11):
        batch_name = f"{dataset}_batch_{batch_id:02d}"
        batch_path = BATCH_DIR / f"{batch_name}.jsonl"

        assert batch_path.is_file()

        for condition in ["unwatermarked", "watermarked"]:
            watermark_enabled = condition == "watermarked"

            config = {
                "model_configuration": MODEL_CONFIG,
                "watermark_type": "Ours" if watermark_enabled else "None",
                "watermark_config": WM_CONFIG if watermark_enabled else NO_WM_CONFIG,
                "evaluation_config": {
                    "num_samples": 20,
                    "batch_size": 1,
                    "evaluation_datasets": [{
                        "path": str(batch_path),
                        "user_prompt": PROMPT_TEMPLATES[dataset],
                        "chat": True,
                        "max_length": 150,
                        "min_length": 1
                    }],
                    "tqdm": True
                }
            }

            filename = f"{batch_name}_{condition}.yaml"
            config_path = CONFIG_DIR / filename

            config_path.write_text(
                yaml.safe_dump(config, sort_keys=False),
                encoding="utf-8"
            )

            print(f"Created: {filename}")

print("Configuration files created.")

Created: 2-1_longform_qa_batch_01_unwatermarked.yaml
Created: 2-1_longform_qa_batch_01_watermarked.yaml
Created: 2-1_longform_qa_batch_02_unwatermarked.yaml
Created: 2-1_longform_qa_batch_02_watermarked.yaml
Created: 2-1_longform_qa_batch_03_unwatermarked.yaml
Created: 2-1_longform_qa_batch_03_watermarked.yaml
Created: 2-1_longform_qa_batch_04_unwatermarked.yaml
Created: 2-1_longform_qa_batch_04_watermarked.yaml
Created: 2-1_longform_qa_batch_05_unwatermarked.yaml
Created: 2-1_longform_qa_batch_05_watermarked.yaml
Created: 2-1_longform_qa_batch_06_unwatermarked.yaml
Created: 2-1_longform_qa_batch_06_watermarked.yaml
Created: 2-1_longform_qa_batch_07_unwatermarked.yaml
Created: 2-1_longform_qa_batch_07_watermarked.yaml
Created: 2-1_longform_qa_batch_08_unwatermarked.yaml
Created: 2-1_longform_qa_batch_08_watermarked.yaml
Created: 2-1_longform_qa_batch_09_unwatermarked.yaml
Created: 2-1_longform_qa_batch_09_watermarked.yaml
Created: 2-1_longform_qa_batch_10_unwatermarked.yaml
Created: 2-

## 5. Text Generation

We use the official `run_config.py` script to generate responses
with LLaDA-8B-Instruct.

Each batch contains 20 prompts and is generated under one
of two conditions: watermarked or unwatermarked.

The complete experiment consists of:

- 3 WaterBench datasets
- 10 batches per dataset
- 20 prompts per batch
- 2 watermark conditions

This results in 1,200 generated responses.

We first perform a smoke test on a single watermarked batch.
The full generation procedure is provided separately.

Results are stored in separate directories and existing
outputs are not overwritten automatically.

In [5]:
import base64
import html
from pathlib import Path
from IPython.display import HTML, display

def download_link(source, filename):
    source = Path(source)

    assert source.is_file(), f"Missing file: {source}"

    encoded = base64.b64encode(source.read_bytes()).decode("ascii")
    safe_name = html.escape(filename, quote=True)

    display(HTML(
        f'<a download="{safe_name}" '
        f'href="data:application/octet-stream;base64,{encoded}">'
        f'Download {safe_name}</a>'
    ))

### 5.1. Smoke Test — Single Batch Generation

We validate the generation pipeline using one batch of
20 Longform QA prompts with watermarking enabled.

The smoke test uses the same model, generation parameters,
and watermark configuration as the main experiment.

Test outputs are saved to `/content/smoke_test_results`
and are excluded from the experimental dataset.

The resulting JSONL file is validated and made available
for local download.

In [ ]:
import subprocess
import json
import time
from pathlib import Path

REPO = Path("/content/diffusion-lm-watermark")
PYTHON = Path("/content/.venv/bin/python")

CONFIG_DIR = Path("/content/production_configs")
RESULT_DIR = Path("/content/smoke_test_results")

# One watermarked batch containing 20 prompts
NAME = "2-1_longform_qa_batch_01_watermarked"

config = CONFIG_DIR / f"{NAME}.yaml"
output = RESULT_DIR / NAME
result_file = output / "results.json"

assert config.is_file(), f"Missing configuration: {config}"

# Avoid overwriting existing results
if result_file.exists():
    print("Existing smoke test found. Skipping generation.")

else:
    if output.exists():
        raise RuntimeError(
            f"Incomplete output directory: {output}"
        )

    output.mkdir(parents=True)

    start = time.time()

    subprocess.run(
        [
            str(PYTHON),
            "-u",
            "scripts/run_config.py",
            "--config", str(config),
            "--output_path", str(output),
            "--num_samples", "20"
        ],
        cwd=REPO,
        check=True
    )

    print(
        f"Generation time: {(time.time() - start) / 60:.1f} min"
    )

# Validate results
assert result_file.is_file(), f"Missing results: {result_file}"

with result_file.open(encoding="utf-8") as f:
    rows = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

assert len(rows) == 20
assert len({row["prompt"] for row in rows}) == 20
assert all(str(row.get("completion", "")).strip() for row in rows)

print("Smoke test successful.")
print("Responses:", len(rows))
print("Unique prompts:", len({row["prompt"] for row in rows}))
print("Output:", result_file)

# Download results to local computer
download_link(result_file, f"{NAME}.jsonl")

### 5.2. Full Dataset Generation

The following cell documents the generation procedure
for the complete experimental dataset.

It iterates over all 30 prompt batches and both watermark
conditions, producing 60 independent generation runs.

Each run generates 20 responses and stores the results
in a dedicated output directory.

Existing results are not overwritten automatically.

The complete dataset was generated incrementally during
the original experiments. This cell is retained for
reproducibility and is disabled by default to prevent
accidental regeneration.

In [ ]:
import subprocess
import json
import time
from pathlib import Path

REPO = Path("/content/diffusion-lm-watermark")
PYTHON = Path("/content/.venv/bin/python")

CONFIG_DIR = Path("/content/production_configs")
RESULT_DIR = Path("/content/production_results")

DATASETS = [
    "2-1_longform_qa",
    "2-2_finance_qa",
    "5-1_alpacafarm"
]

BATCH_IDS = range(1, 11)
CONDITIONS = ["unwatermarked", "watermarked"]

# Safety switch: do not regenerate the original dataset
RUN_FULL_GENERATION = False

def run_full_generation():
    RESULT_DIR.mkdir(parents=True, exist_ok=True)

    for dataset in DATASETS:
        for condition in CONDITIONS:
            for batch_id in BATCH_IDS:

                name = (
                    f"{dataset}_batch_{batch_id:02d}_{condition}"
                )

                config = CONFIG_DIR / f"{name}.yaml"
                output = RESULT_DIR / name
                result_file = output / "results.json"

                assert config.is_file(), f"Missing config: {config}"

                print(f"\nProcessing: {name}", flush=True)

                if result_file.exists():
                    print("Existing results found. Skipping generation.")

                else:
                    if output.exists():
                        raise RuntimeError(
                            f"Incomplete output directory: {output}"
                        )

                    output.mkdir(parents=True)

                    start = time.time()

                    subprocess.run(
                        [
                            str(PYTHON),
                            "-u",
                            "scripts/run_config.py",
                            "--config", str(config),
                            "--output_path", str(output),
                            "--num_samples", "20"
                        ],
                        cwd=REPO,
                        check=True
                    )

                    print(
                        f"Generation time: "
                        f"{(time.time() - start) / 60:.1f} min",
                        flush=True
                    )

                # Validate results
                assert result_file.is_file()

                with result_file.open(encoding="utf-8") as f:
                    rows = [
                        json.loads(line)
                        for line in f
                        if line.strip()
                    ]

                n_rows = len(rows)
                n_unique = len({row["prompt"] for row in rows})

                print(f"Responses: {n_rows}")
                print(f"Unique prompts: {n_unique}/20")

                valid = (
                    n_rows == 20
                    and n_unique == 20
                    and all(
                        str(row.get("completion", "")).strip()
                        for row in rows
                    )
                )

                filename = (
                    f"{name}.jsonl"
                    if valid
                    else f"{name}_RAW.jsonl"
                )

                download_link(result_file, filename)

                if not valid:
                    raise RuntimeError(
                        f"Invalid results: {name}. "
                        "Raw outputs preserved."
                    )

    print("\nFull dataset generation completed.")


if RUN_FULL_GENERATION:
    run_full_generation()
else:
    print("Full generation disabled. No model execution.")

## 6. Results Validation

After generation, we verify the following properties:

- Exactly 20 responses per batch
- Exactly 20 unique prompts per batch
- No empty completions
- Valid JSONL formatting
- Matching prompts between the two experimental conditions

Unexpected files are preserved for manual inspection.

The original generation outputs are kept unchanged.

This validation is intended for the complete set of original
generation outputs and requires all 60 result files to be available.

In [ ]:
import json
from pathlib import Path

RESULT_DIR = Path("/content/production_results")

DATASETS = [
    "2-1_longform_qa",
    "2-2_finance_qa",
    "5-1_alpacafarm"
]

CONDITIONS = ["unwatermarked", "watermarked"]

all_valid = True

for dataset in DATASETS:
    for batch_id in range(1, 11):

        prompts_by_condition = {}

        for condition in CONDITIONS:
            name = f"{dataset}_batch_{batch_id:02d}_{condition}"
            path = RESULT_DIR / name / "results.json"

            if not path.is_file():
                print(f"MISSING: {name}")
                all_valid = False
                continue

            with path.open(encoding="utf-8") as f:
                rows = [
                    json.loads(line)
                    for line in f
                    if line.strip()
                ]

            prompts = {row["prompt"] for row in rows}

            valid = (
                len(rows) == 20
                and len(prompts) == 20
                and all(
                    str(row.get("completion", "")).strip()
                    for row in rows
                )
            )

            prompts_by_condition[condition] = prompts
            all_valid = all_valid and valid

            print(f"{name}: {len(rows)} responses, valid={valid}")

        if len(prompts_by_condition) == 2:
            matching = (
                prompts_by_condition["watermarked"]
                == prompts_by_condition["unwatermarked"]
            )

            if not matching:
                print(f"PROMPT MISMATCH: {dataset} batch {batch_id:02d}")

            all_valid = all_valid and matching

print("\nAll checks passed:", all_valid)

## 7. Results Storage and Reproducibility

Google Colab uses temporary storage. Files saved under `/content/`
may disappear when the runtime is terminated.

Generated JSONL files are downloaded to the local computer using
the `download_link()` utility before disconnecting the runtime.

The complete experiment consists of:

- 3 WaterBench datasets
- 200 prompts per dataset
- 2 generation conditions
- 1,200 generated responses

The original dataset was generated incrementally across multiple
Colab sessions, using the same experimental configurations.

The smoke test is stored separately and is not included in the
experimental results.

Subsequent data preparation, watermark detection, and attack
evaluation are performed locally without requiring a generation GPU.